In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pm4py

In [ ]:
PROJECT_ROOT = Path("../").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "BPI Challenge 2017.xes"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())

In [ ]:
log = pm4py.read_xes(str(DATA_PATH))
df = pm4py.convert_to_dataframe(log)

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(col)

df.dtypes

In [ ]:
case_col = "case:concept:name"
activity_col = "concept:name"
timestamp_col = "time:timestamp"
resource_col = "org:resource"

df[[case_col, activity_col, timestamp_col]].head(20)

In [ ]:
df[[case_col, activity_col, timestamp_col, resource_col]].head(20)

In [ ]:
num_cases = df[case_col].nunique()
num_events = len(df)
num_activities = df[activity_col].nunique()

print("Number of cases:", num_cases)
print("Number of events:", num_events)
print("Number of unique activities:", num_activities)

if resource_col in df.columns:
    print("Number of resources:", df[resource_col].nunique())

In [ ]:
activity_counts = df[activity_col].value_counts()

activity_counts.head(20)

In [ ]:
case_lengths = df.groupby(case_col).size()

case_lengths.describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(case_lengths, bins=50)
plt.title("Case Length Distribution")
plt.xlabel("Number of events per case")
plt.ylabel("Number of cases")
plt.tight_layout()
plt.show()

In [ ]:
variants = (
    df.sort_values([case_col, timestamp_col])
      .groupby(case_col)[activity_col]
      .apply(lambda x: " -> ".join(x.astype(str)))
)

variant_counts = variants.value_counts()

print("Number of variants:", variant_counts.shape[0])
variant_counts.head(10)

In [ ]:
# Ensure timestamp column is datetime
df[timestamp_col] = pd.to_datetime(df[timestamp_col])

# For each case, get first and last timestamp
case_times = df.groupby(case_col)[timestamp_col].agg(["min", "max"])

# Duration = last event time - first event time
case_times["duration"] = case_times["max"] - case_times["min"]
case_times["duration_seconds"] = case_times["duration"].dt.total_seconds()
case_times["duration_minutes"] = case_times["duration_seconds"] / 60
case_times["duration_days"] = case_times["duration_seconds"] / (24 * 3600)

case_duration_summary = case_times[
    ["duration_seconds", "duration_minutes", "duration_days"]
].describe()

case_duration_summary

In [ ]:
case_attribute_cols = [col for col in df.columns if col.startswith("case:")]
event_attribute_cols = [col for col in df.columns if not col.startswith("case:")]

print("Case attributes:")
for col in case_attribute_cols:
    print("-", col)

print("\nEvent attributes:")
for col in event_attribute_cols:
    print("-", col)

print("\nNumber of case attribute labels:", len(case_attribute_cols))
print("Number of event attribute labels:", len(event_attribute_cols))

In [ ]:
from pandas.api.types import is_string_dtype, is_object_dtype, is_categorical_dtype

categorical_event_cols = [
    col for col in event_attribute_cols
    if is_string_dtype(df[col]) or is_object_dtype(df[col]) or is_categorical_dtype(df[col])
]

print("Categorical event attributes:")
for col in categorical_event_cols:
    print("-", col)

print("\nNumber of categorical event attributes:", len(categorical_event_cols))

In [ ]:
event_origin_counts = df["EventOrigin"].value_counts(dropna=False)
event_origin_counts

In [ ]:
loan_goal_counts = df[[case_col, "case:LoanGoal"]].drop_duplicates()[ "case:LoanGoal"].value_counts(dropna=False)
loan_goal_counts

In [ ]:
top_activities = df[activity_col].value_counts().head(15)

plt.figure(figsize=(10, 6))
top_activities.sort_values().plot(kind="barh")
plt.title("Top 15 Activities by Frequency")
plt.xlabel("Number of Events")
plt.ylabel("Activity")
plt.tight_layout()
plt.show()

In [ ]:
case_lengths = df.groupby(case_col).size()

plt.figure(figsize=(10, 6))
plt.hist(case_lengths, bins=50)
plt.title("Case Length Distribution")
plt.xlabel("Number of Events per Case")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(case_times["duration_days"], bins=50)
plt.title("Case Duration Distribution")
plt.xlabel("Case Duration in Days")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
event_origin_counts.plot(kind="bar")
plt.title("Event Origin Distribution")
plt.xlabel("Event Origin")
plt.ylabel("Number of Events")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()